In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [2]:
data = pd.read_csv('fraudTest.csv')

# One hot encoding

In [3]:
onehot = data.drop(columns=['Unnamed: 0', 'cc_num','merchant', 'category','first','last','street','lat','long','city','job','merch_lat','merch_long','state','zip','trans_num','unix_time'])

In [4]:
onehot['trans_date_trans_time']=pd.to_datetime(onehot['trans_date_trans_time'])
onehot['dob']=pd.to_datetime(onehot['dob'])

In [5]:
onehot_encoded = pd.DataFrame()

In [6]:
amt_bins=[0,50,100,500,1000,5000,10000,50000,100000,500000,1000000,1000000000000]
amt_labels=['0-50','50-100','100-500','500-1000','1000-5000','5000-10000','10000-50000','50000-100000','100000-500000','500000-1000000','1000000+']
onehot_amt =pd.DataFrame( pd.cut(onehot['amt'],bins=amt_bins, labels=amt_labels))

In [7]:
city_pop_bins=[0,1000, 50000, 100000, 500000, 1000000, 10000000000]
city_pop_labels=['0-1000','1000-50000','50000-100000','100000-500000','500000-1000000','1000000+']
onehot_city_pop = pd.DataFrame(pd.cut(onehot['city_pop'], bins=city_pop_bins, labels=city_pop_labels))

In [8]:
onehot["y.o."] = (pd.Timestamp.today() - onehot["dob"]).dt.days // 365.25
yo_bins = [0,16,25,35,50,65,80,100,1000]
yo_labels = ['0-16','16-25','25-35','35-50','50-65','65-80','80-100','100+']
onehot_yo = pd.DataFrame(pd.cut(onehot['y.o.'], bins = yo_bins, labels = yo_labels))

In [9]:
onehot_gender = pd.DataFrame(onehot['gender'])

In [10]:
onehot['trans_date_trans_time'] = pd.to_datetime(onehot['trans_date_trans_time']) 
onehot['time'] = onehot['trans_date_trans_time'].dt.hour
onehot['day_of_the_week'] = onehot['trans_date_trans_time'].dt.dayofweek

In [11]:
day_bins = [0,5,6]
day_labels=['Weekdays', 'Weekend']
onehot_day = pd.DataFrame(pd.cut(onehot['day_of_the_week'], bins=day_bins, labels=day_labels))

In [12]:
time_bins=[0,6,12,18,24]
time_labels=['0-6','6-12','12-18','18-24']
onehot_time = pd.DataFrame(pd.cut(onehot['time'], bins=time_bins, labels=time_labels))

In [13]:
onehot_encoder = OneHotEncoder(handle_unknown='ignore')

# Target encoding

In [14]:
target_example = data.drop(columns=['trans_date_trans_time','Unnamed: 0', 'amt', 'gender', 'cc_num','city_pop','dob','trans_num','lat','long','merch_lat','merch_long','street','unix_time','zip','first','last','city'])

In [15]:
target_example['first_last'] = data['first']+' '+data['last']

In [16]:
merchant_encoding = target_example.groupby('merchant')['is_fraud'].agg(mean='mean')
target_merchant = pd.DataFrame(target_example['merchant'].map(merchant_encoding['mean']))

In [17]:
category_encoding = target_example.groupby('category')['is_fraud'].agg(mean='mean')
target_category = pd.DataFrame(target_example['category'].map(category_encoding['mean']))

In [18]:
first_last_encoding = target_example.groupby('first_last')['is_fraud'].agg(mean='mean')
target_first_last = pd.DataFrame(target_example['first_last'].map(first_last_encoding['mean']))

In [19]:
state_encoding = target_example.groupby('state')['is_fraud'].agg(mean='mean')
target_state = pd.DataFrame(target_example['state'].map(state_encoding['mean']))

In [20]:
job_encoding = target_example.groupby('job')['is_fraud'].agg(mean='mean')
target_job = pd.DataFrame(target_example['job'].map(job_encoding['mean']))

# Grouping 

In [21]:
full_onehot_encoder = [onehot_amt,onehot_city_pop,onehot_yo,onehot_gender, onehot_day,onehot_time]
full_onehot_encoded = []
full_target_encoder = [target_merchant,target_category,target_first_last,target_state,target_job]

In [22]:
for i in full_onehot_encoder:
    full_onehot_encoded.append(onehot_encoder.fit_transform(i))